In [ ]:
%pip install mlflow -qq

In [ ]:
!uv pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

In [ ]:
import sys

sys.path.append('/kaggle/input/datasets/maksimbessolitsyn/')

In [ ]:
import logging
import warnings
import os

warnings.filterwarnings("ignore", category=UserWarning, module=r"torch(\.|$)")
warnings.filterwarnings("ignore", category=FutureWarning, module=r"torch(\.|$)")
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)




In [ ]:
LOG_DIR = "./mlruns"


In [ ]:
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")
SEED = 42

In [ ]:
from sasrec import run_ddp_training, ExperimentConfig

In [ ]:
seeds = [42, 43, 44, 45, 46]

In [ ]:
import torch

In [ ]:
constant = ExperimentConfig(
    graph=ExperimentConfig.GraphConfig(
        n_layers=4,
        d_model=256,
        n_heads=4,
        dropout=0.0,
        log_q_correction=1.0,
        is_cosine_similarity=True,
    ),
    data=ExperimentConfig.DataConfig(
        vocab_size=157_162,
        max_seq_len=100,
        bos=0,
        path_interactions=PATH_INTERACTIONS,
        path_embeddings=PATH_EMBEDDINGS,
        path_artists=PATH_ARTISTS,
        core_min_interaction_per_user=5,
        test_interval_seconds=7 * 24 * 60 * 60,
        max_train_events_per_user=100,
    ),
    tau=ExperimentConfig.TauConfig(
        class_name="ConstantTau",
        json_args={
            "initial_tau": 0.45,
            "tau_min": 0.04,
            "tau_max": 0.05,
            "num_epochs": 15,
            "num_tokens_per_epoch": 4_019_032,
        },
    ),
    training_dataset=ExperimentConfig.TrainingDatasetConfig(
        batch_size=128,
        device="cuda",
        chunk_rows=64000,
        shuffle=True,
        seed=42,
        pin_memory=True,
        uniform_negative_items=22_000,
        in_batch_negative_items=8_000,
    ),
    test_dataset=ExperimentConfig.TestDatasetConfig(
        batch_size=32,
        device="cuda",
    ),
    optimizer=ExperimentConfig.OptimizerConfig(
        class_name="AdamW",
        json_args={
            "lr": 3e-3,
            "weight_decay": 1e-5,
        },
    ),
    scheduler=ExperimentConfig.SchedulerConfig(
        class_name=None,
        json_args={},
    ),
    training=ExperimentConfig.TrainingConfig(
        num_epochs=15,
        grad_clip=1.0,
        eval_every=1,
        logging=True,
        log_dir=LOG_DIR,
        console_logging=False,
        seed=None,
    ),
    evaluator=ExperimentConfig.EvaluatorConfig(
        topk=100,
    ),
)

metrics = {}

for seed in seeds:

    metrics[seed] = run_ddp_training(
        constant, 
        torch.cuda.device_count(),
    )

print(constant.build_tau().experiment_name())
print("AVG Metrics:")

for metric in ["recall", "ndcg", "hitrate", "coverage"]:

    values = [metrics[seed][metric] for seed in seeds]
    avg = sum(values) / len(values)
    disp = sum([(metrics[seed][metric] - avg)**2 for seed in seeds])

    print(f"{metric}: {avg=}, {disp=}")

In [ ]:
cos_per_user = ExperimentConfig(
    graph=ExperimentConfig.GraphConfig(
        n_layers=4,
        d_model=256,
        n_heads=4,
        dropout=0.0,
        log_q_correction=1.0,
        is_cosine_similarity=True,
    ),
    data=ExperimentConfig.DataConfig(
        vocab_size=157_162,
        max_seq_len=100,
        bos=0,
        path_interactions=PATH_INTERACTIONS,
        path_embeddings=PATH_EMBEDDINGS,
        path_artists=PATH_ARTISTS,
        core_min_interaction_per_user=5,
        test_interval_seconds=7 * 24 * 60 * 60,
        max_train_events_per_user=100,
    ),
    tau=ExperimentConfig.TauConfig(
        class_name="CosPerUserTau",
        json_args={
            "initial_tau": 0.45,
            "tau_min": 0.04,
            "tau_max": 0.05,
            "num_epochs": 15,
            "num_tokens_per_epoch": 4_019_032,
        },
    ),
    training_dataset=ExperimentConfig.TrainingDatasetConfig(
        batch_size=128,
        device="cuda",
        chunk_rows=64000,
        shuffle=True,
        seed=42,
        pin_memory=True,
        uniform_negative_items=22_000,
        in_batch_negative_items=8_000,
    ),
    test_dataset=ExperimentConfig.TestDatasetConfig(
        batch_size=32,
        device="cuda",
    ),
    optimizer=ExperimentConfig.OptimizerConfig(
        class_name="AdamW",
        json_args={
            "lr": 3e-3,
            "weight_decay": 1e-5,
        },
    ),
    scheduler=ExperimentConfig.SchedulerConfig(
        class_name=None,
        json_args={},
    ),
    training=ExperimentConfig.TrainingConfig(
        num_epochs=15,
        grad_clip=1.0,
        eval_every=1,
        logging=True,
        log_dir=LOG_DIR,
        console_logging=False,
        seed=None,
    ),
    evaluator=ExperimentConfig.EvaluatorConfig(
        topk=100,
    ),
)

metrics = {}

for seed in seeds:

    metrics[seed] = run_ddp_training(
        cos_per_user, 
        torch.cuda.device_count(),
    )

print(cos_per_user.build_tau().experiment_name())
print("AVG Metrics:")

for metric in ["recall", "ndcg", "hitrate", "coverage"]:

    values = [metrics[seed][metric] for seed in seeds]
    avg = sum(values) / len(values)
    disp = sum([(metrics[seed][metric] - avg)**2 for seed in seeds])

    print(f"{metric}: {avg=}, {disp=}")